# Agentic RAG System with OpenAI + LangChain + ChromaDB

A beginner-friendly but standards-aligned notebook for learning, building, and extending a real PDF-based RAG system.


## What this notebook teaches

This notebook is designed as a **learn-by-building project**.

You will learn:

- what RAG is and why it matters
- when to use **simple RAG** vs **agentic RAG**
- how to load **real PDF files**
- how to split text into chunks properly
- how embeddings work at a practical level
- how to store vectors in **ChromaDB**
- how to make the database **persistent on disk**
- how to retrieve relevant chunks
- how to generate grounded answers from retrieved context
- how to add a lightweight **agentic layer**
- how to debug common issues

---

## Project outcome

By the end, you will have a working project that can:

1. read PDFs from a local folder  
2. convert them into embeddings  
3. save them in ChromaDB  
4. reload the vector database later without re-indexing  
5. answer questions from your documents  
6. use a simple agentic workflow:
   - rewrite the question
   - retrieve context
   - judge relevance
   - answer from evidence only

---

## Good use cases

This exact architecture is useful for:

- resume screening assistant
- policy and SOP Q&A bot
- legal or compliance document search
- academic paper assistant
- company research assistant
- interview preparation helper
- support knowledge base
- internal enterprise document chatbot

---

## Recommended folder structure

```text
agentic-rag-project/
│
├── data/
│   └── pdfs/
│       ├── sample1.pdf
│       ├── sample2.pdf
│
├── storage/
│   └── chroma_db/
│
├── notebooks/
│   └── agentic_rag_system.ipynb
│
├── .env
├── requirements.txt
└── app.py
```



## Standards and design choices used here

This notebook follows practical project standards:

- **separate data folder** for source documents
- **persistent vector database** in a storage folder
- **environment variable** for API key
- **modular functions** instead of one long script
- **clear debug prints** for troubleshooting
- **retrieval before generation**
- **answer only from retrieved context**
- **agentic validation step** before final answer

This is not a full autonomous agent. It is a **simple agentic RAG system**, which is often the right place to start.



## Important concepts before coding

### 1. What is RAG?

**RAG** means **Retrieval-Augmented Generation**.

Instead of asking the language model to answer from memory alone, we first retrieve relevant information from your own documents and then ask the model to answer using that retrieved content.

Basic flow:

```text
User Question
   ↓
Retriever finds relevant chunks
   ↓
LLM reads retrieved context
   ↓
Grounded answer
```

### 2. Why is RAG useful?

RAG helps:

- reduce hallucination
- answer from private company data
- use up-to-date documents
- avoid fine-tuning for every knowledge update

### 3. What is Agentic RAG?

A normal RAG pipeline is:

```text
Question → Retrieve → Answer
```

A simple agentic RAG pipeline is:

```text
Question → Rewrite → Retrieve → Check relevance → Answer
```

This makes the system more reliable because it can improve the query and verify whether retrieved content is useful.

### 4. What is an embedding?

An embedding is a numeric vector representation of text.  
Similar texts get vectors that are close to each other in vector space.

### 5. What is ChromaDB?

ChromaDB is a vector database that stores embeddings and lets you search semantically.

### 6. Why persistence matters

If you do not persist the vector database, it may be lost when your notebook restarts.

That is why we will use a dedicated path like:

```python
PERSIST_DIR = "./storage/chroma_db"
```



## Install dependencies

Run this cell once.


In [ ]:
!pip install -U langchain langchain-openai langchain-chroma langchain-community chromadb pypdf python-dotenv jupyter


## Environment setup

Create a `.env` file in your project root with:

```text
OPENAI_API_KEY=your_openai_api_key_here
```

Do not hardcode secrets in production code.


In [ ]:

import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Put it in your .env file.")

print("API key loaded successfully.")



## Define key paths

These two paths matter most:

- `PDF_DIR` = where your real PDFs are stored
- `PERSIST_DIR` = where ChromaDB will persist files

Put your PDFs in `./data/pdfs`.


In [ ]:

from pathlib import Path

PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "data" / "pdfs"
PERSIST_DIR = PROJECT_ROOT / "storage" / "chroma_db"

PDF_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF directory:", PDF_DIR)
print("Persist directory:", PERSIST_DIR)
print("Persist directory absolute path:", PERSIST_DIR.resolve())



## Debug cell: inspect your working paths

This cell is useful when something is not loading correctly.


In [ ]:

print("Current working directory:", os.getcwd())
print("PDF folder exists:", PDF_DIR.exists())
print("Persist folder exists:", PERSIST_DIR.exists())
print("PDF files present:", [p.name for p in PDF_DIR.glob("*.pdf")])



## Step 1: Load real PDFs

We will use `PyPDFLoader` to load each PDF file.

Each loaded page becomes a LangChain `Document` object with:
- `page_content`
- `metadata`

This metadata is extremely useful because it can store:
- source file path
- page number
- document identity


In [ ]:

from langchain_community.document_loaders import PyPDFLoader

def load_pdfs_from_folder(pdf_dir: Path):
    all_docs = []
    pdf_files = list(pdf_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in: {pdf_dir}")
        return all_docs

    for pdf_file in pdf_files:
        print(f"Loading PDF: {pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        all_docs.extend(docs)

    return all_docs

raw_docs = load_pdfs_from_folder(PDF_DIR)
print("Total loaded pages/documents:", len(raw_docs))



## Inspect a loaded document

This helps you understand the structure before chunking.


In [ ]:

if raw_docs:
    print("Sample content preview:")
    print(raw_docs[0].page_content[:1000])
    print("\nMetadata:")
    print(raw_docs[0].metadata)
else:
    print("No documents loaded yet. Add PDFs into the data/pdfs folder first.")



## Step 2: Split documents into chunks

### Why chunking is needed

LLMs and retrievers work better on smaller, meaningful pieces of text than on entire long files.

### Good chunking principles

- chunks should be big enough to contain meaning
- chunks should not be too large
- overlap helps preserve continuity
- different document types may need different chunk sizes

### Suggested starting values

- `chunk_size = 800 to 1200`
- `chunk_overlap = 100 to 200`

For PDFs, a practical starter configuration is:

- `chunk_size = 1000`
- `chunk_overlap = 200`


In [ ]:

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(raw_docs)

print("Total chunks created:", len(split_docs))


In [ ]:

if split_docs:
    print("Chunk preview:")
    print(split_docs[0].page_content[:800])
    print("\nChunk metadata:")
    print(split_docs[0].metadata)



## Step 3: Create embeddings

We will use OpenAI embeddings.

### Practical note

A smaller embedding model is usually a good learning default because it is simpler and more cost-conscious.

### What happens here?

Every chunk is converted into a vector.

Later, when the user asks a question, the question is also embedded.  
Then the retriever finds chunks whose vectors are most similar to the query vector.


In [ ]:

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model ready.")



## Step 4: Build and persist the Chroma vector database

This is one of the most important steps.

If `persist_directory` is provided, Chroma stores the vector database on disk so that you can reload it later.

This avoids repeating the full embedding process every time the notebook restarts.


In [ ]:

from langchain_chroma import Chroma

COLLECTION_NAME = "agentic_rag_pdf_collection"

def build_and_persist_vectorstore(documents, embedding_model, persist_dir, collection_name):
    if not documents:
        raise ValueError("No documents found to index. Please load PDFs first.")

    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=str(persist_dir),
        collection_name=collection_name
    )
    return vectorstore

vectorstore = build_and_persist_vectorstore(
    documents=split_docs,
    embedding_model=embedding_model,
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME
)

print("Vector store created.")
print("Persistence path:", PERSIST_DIR.resolve())



## Inspect persisted files

This helps you see where the vector database is stored physically.


In [ ]:

for root, dirs, files in os.walk(PERSIST_DIR):
    level = root.replace(str(PERSIST_DIR), "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")



## Step 5: Reload the vector database later

This is what you will usually do after the database is already built once.

Instead of re-reading and re-embedding every PDF again, you can reconnect to the same persistent vector database.


In [ ]:

def load_existing_vectorstore(embedding_model, persist_dir, collection_name):
    return Chroma(
        embedding_function=embedding_model,
        persist_directory=str(persist_dir),
        collection_name=collection_name
    )

reloaded_vectorstore = load_existing_vectorstore(
    embedding_model=embedding_model,
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME
)

print("Reloaded vector store from disk successfully.")



## Step 6: Create a retriever

A retriever converts the vector database into a search interface.

We will retrieve the top `k` matching chunks.

### Good default

Start with:

```python
search_kwargs={"k": 3}
```

Then tune later.


In [ ]:

retriever = reloaded_vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready.")



## Try retrieval only

Before involving the LLM, always test retrieval by itself.

This is a best practice.

If retrieval is bad, the final answers will also be bad.


In [ ]:

test_query = "Summarize the main topics discussed in the PDF."

retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} chunks.")
for i, doc in enumerate(retrieved_docs, start=1):
    print("\n" + "="*80)
    print(f"Result {i}")
    print("- Metadata:", doc.metadata)
    print("- Content preview:")
    print(doc.page_content[:700])



## Step 7: Create the LLM

We now use a chat model for reasoning over retrieved text.


In [ ]:

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("LLM ready.")



## Step 8: Build a simple RAG function

This is the classic baseline RAG system.

### Baseline flow

1. retrieve top chunks
2. combine them into context
3. send prompt to model
4. ask model to answer only from context

This baseline is important because you should always compare your advanced pipeline against a simple baseline.


In [ ]:

def format_context(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def simple_rag(query: str, retriever, llm):
    docs = retriever.invoke(query)
    context = format_context(docs)

    prompt = f'''
You are a helpful assistant.
Answer the question using only the provided context.
If the answer is not in the context, say you do not have enough information.

Context:
{context}

Question:
{query}
'''

    response = llm.invoke(prompt)
    return {
        "query": query,
        "retrieved_docs": docs,
        "answer": response.content
    }

result = simple_rag("What are the main ideas in these PDFs?", retriever, llm)
print(result["answer"])



## Why simple RAG may still fail

Even good RAG systems can fail because:

- the original query is vague
- retrieval pulls weak chunks
- chunks are too large or too small
- documents are noisy
- top-k is not tuned
- the model answers too aggressively

This is why people add an **agentic layer**.



# Agentic RAG section

The goal here is not to build a giant autonomous agent.

The goal is to add a few useful reasoning steps:

1. **rewrite the question** for better retrieval  
2. **retrieve context**  
3. **check whether context is relevant**  
4. **answer only if context seems sufficient**

This is a lightweight but practical agentic architecture.



## Step 9: Query rewriting

### Why rewrite?

Users often ask vague questions like:

- "tell me about this"
- "what does it say"
- "give summary"
- "explain revenue"

A rewritten query can be more explicit and retrieval-friendly.


In [ ]:

def rewrite_query(query: str, llm):
    prompt = f'''
You are helping improve document retrieval.
Rewrite the user's question so it is clearer, more specific, and retrieval-friendly.
Do not answer the question.
Only return the improved query.

Original question:
{query}
'''
    return llm.invoke(prompt).content.strip()

rewritten = rewrite_query("tell me the important things from the file", llm)
print("Rewritten query:", rewritten)



## Step 10: Relevance grading

### Why grade relevance?

Sometimes retrieval returns chunks that are only loosely related.

A relevance grader helps decide whether the system should answer confidently or say it needs better evidence.


In [ ]:

def grade_relevance(query: str, docs, llm):
    context = format_context(docs)

    prompt = f'''
You are judging whether the retrieved context is relevant enough to answer the user's question.

Question:
{query}

Retrieved context:
{context}

Reply with only one word:
YES
or
NO
'''
    result = llm.invoke(prompt).content.strip().upper()
    return "YES" if "YES" in result else "NO"

relevance = grade_relevance("Summarize the main points from the PDFs", retrieved_docs, llm)
print("Relevance decision:", relevance)



## Step 11: Grounded answer generation

This function answers only from the retrieved evidence and falls back safely if the evidence is weak.


In [ ]:

def generate_grounded_answer(query: str, docs, llm):
    context = format_context(docs)

    prompt = f'''
You are a careful AI assistant.

Use only the retrieved context below.
Do not invent facts.
If the answer is not supported by the context, say:
"I do not have enough information in the retrieved documents."

Retrieved context:
{context}

Question:
{query}
'''
    return llm.invoke(prompt).content



## Step 12: Final agentic RAG pipeline

This is the main pipeline for the notebook.

### Flow

```text
User question
   ↓
Rewrite question
   ↓
Retrieve chunks
   ↓
Check relevance
   ↓
If relevant → answer
Else → decline safely
```


In [ ]:

def agentic_rag(query: str, retriever, llm):
    improved_query = rewrite_query(query, llm)
    docs = retriever.invoke(improved_query)
    relevance = grade_relevance(improved_query, docs, llm)

    if relevance == "NO":
        return {
            "original_query": query,
            "improved_query": improved_query,
            "relevance": relevance,
            "retrieved_docs": docs,
            "answer": "I do not have enough relevant information in the retrieved documents."
        }

    answer = generate_grounded_answer(improved_query, docs, llm)

    return {
        "original_query": query,
        "improved_query": improved_query,
        "relevance": relevance,
        "retrieved_docs": docs,
        "answer": answer
    }



## Test the agentic pipeline


In [ ]:

agentic_result = agentic_rag(
    query="Give me the key points from the uploaded PDFs",
    retriever=retriever,
    llm=llm
)

print("Original query:", agentic_result["original_query"])
print("Improved query:", agentic_result["improved_query"])
print("Relevance:", agentic_result["relevance"])
print("\nFinal answer:\n")
print(agentic_result["answer"])



## Show retrieved evidence for transparency

In production-style systems, you often want to show supporting chunks or citations to users.


In [ ]:

for i, doc in enumerate(agentic_result["retrieved_docs"], start=1):
    print("\n" + "="*80)
    print(f"Evidence chunk {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:700])



# Real examples and use cases

## Use case 1: Resume screening assistant

### Problem
You have many resumes in PDF format and want to answer questions like:
- which candidate has Selenium experience?
- who has Playwright and API automation?
- which profile matches SDET + GenAI?

### How this notebook helps
- load resumes as PDFs
- index them in Chroma
- query them semantically

### Example questions
- "Which resume mentions Playwright with TypeScript?"
- "Find candidates with API testing and automation framework design."
- "Summarize the strongest candidate for an SDET role."

---

## Use case 2: Policy and SOP assistant

### Problem
Employees need quick answers from internal policy documents.

### Example questions
- "What is the leave approval process?"
- "What does the reimbursement policy say about travel?"
- "What documents are required for audit review?"

---

## Use case 3: Academic paper assistant

### Problem
Researchers want quick summaries from papers.

### Example questions
- "What is the paper's main contribution?"
- "What methods were used?"
- "What are the limitations?"

---

## Use case 4: Financial or annual report analysis

### Example questions
- "What are the main risks discussed in the annual report?"
- "What does management say about revenue growth?"
- "What ESG themes appear repeatedly?"



# Best practices

## 1. Always inspect retrieval first
Do not blame the LLM immediately. Poor answers often come from weak retrieval.

## 2. Keep chunk size realistic
Too large:
- noisy retrieval
- weak relevance

Too small:
- missing context
- fragmented meaning

## 3. Store metadata
Metadata like file name and page number makes results explainable.

## 4. Persist vector databases
This saves time and money.

## 5. Keep temperature low for factual Q&A
For grounded document Q&A, `temperature=0` is usually a good default.

## 6. Add safe fallback behavior
When the answer is not in the documents, say so.

## 7. Separate indexing from querying
In larger projects:
- one script builds the index
- another script runs Q&A

## 8. Evaluate with real user questions
A RAG system is only as good as its performance on realistic queries.



# Common mistakes

- forgetting to add PDFs to the folder
- forgetting the `.env` file
- not using `persist_directory`
- using huge chunk sizes without testing
- not checking retrieval results manually
- asking the model to answer without grounding
- expecting agentic RAG to fix poor source documents



# Troubleshooting guide


In [ ]:

def debug_environment():
    print("cwd:", Path.cwd())
    print("PDF_DIR exists:", PDF_DIR.exists())
    print("PERSIST_DIR exists:", PERSIST_DIR.exists())
    print("PDF files:", [p.name for p in PDF_DIR.glob("*.pdf")])
    print("Persist absolute path:", PERSIST_DIR.resolve())

debug_environment()



## If no PDFs are loaded

Check:

- did you actually put PDFs into `data/pdfs`?
- are the files real `.pdf` files?
- is the notebook running from the project root you expect?

## If vector DB seems empty

Check:

- were `raw_docs` loaded?
- were `split_docs` created?
- did `Chroma.from_documents(...)` run successfully?
- does the persistence folder contain files?

## If answers are weak

Try:

- better chunk size
- better overlap
- improved query rewriting
- larger `k`
- cleaner PDFs
- better prompts



# Optional improvement ideas

Once this notebook works, you can extend it with:

- conversation memory
- citation formatting
- parent-child retrieval
- multi-query retrieval
- reranking
- hybrid search
- LangGraph orchestration
- Streamlit UI
- API deployment
- evaluation dataset and scoring



# Mini exercises for practice

### Exercise 1
Add 3 different PDFs and compare retrieval quality.

### Exercise 2
Change chunk size from `1000` to `700` and check whether answers improve.

### Exercise 3
Change `k` from `3` to `5` and inspect the effect.

### Exercise 4
Modify the relevance grader to return:
- HIGH
- MEDIUM
- LOW

### Exercise 5
Display source file names and page numbers in the final answer.



# Production-style next steps

A strong real-world next version of this project would split code like this:

- `index_pdfs.py` → reads PDFs and builds Chroma index
- `qa_app.py` → loads persisted Chroma and answers questions
- `utils.py` → helper functions
- `prompts.py` → prompt templates
- `config.py` → path and model settings

That structure makes the project easier to maintain and deploy.



# Final takeaway

You now have a full learning path and working implementation for a **simple agentic RAG system**.

This notebook teaches the correct progression:

1. understand RAG  
2. build a simple baseline  
3. add persistence  
4. test retrieval  
5. add a small agentic layer  
6. improve only after the baseline works

That is the right professional approach.



# Quick reference summary

## Core objects used

- `PyPDFLoader` → load PDFs
- `RecursiveCharacterTextSplitter` → chunk text
- `OpenAIEmbeddings` → create vectors
- `Chroma` → vector DB
- `Retriever` → search similar chunks
- `ChatOpenAI` → answer from context

## Main functions in this notebook

- `load_pdfs_from_folder(...)`
- `build_and_persist_vectorstore(...)`
- `load_existing_vectorstore(...)`
- `simple_rag(...)`
- `rewrite_query(...)`
- `grade_relevance(...)`
- `generate_grounded_answer(...)`
- `agentic_rag(...)`
